# TrOCR Fine-Tuning on the Bentham Dataset

The [Bentham dataset (BenthamR0)](http://transcriptorium.eu/~tsdata/BenthamR0/) contains handwritten historical documents from Jeremy Bentham (18th–19th century). We fine-tune `microsoft/trocr-base-handwritten` on cropped line images paired with ground-truth transcriptions.

## Imports

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

from PIL import Image
from zipfile import ZipFile
from urllib.request import urlretrieve
from dataclasses import dataclass
from torch.utils.data import Dataset
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
)

plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Configuration

In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    BATCH_SIZE:    int   = 2
    GRAD_ACCUM:    int   = 8        # 2 x 8 = effective batch size of 16
    EPOCHS:        int   = 10       # increased from 3 — more passes over data
    LEARNING_RATE: float = 3e-5     # slightly lower — better for the larger base model

@dataclass(frozen=True)
class DatasetConfig:
    DATA_ROOT:          str = "../data/Bentham/BenthamDatasetR0-GT"
    IMAGES_DIR:         str = "../data/Bentham/BenthamDatasetR0-GT/Images/Lines"
    TRANSCRIPTIONS_DIR: str = "../data/Bentham/BenthamDatasetR0-GT/Transcriptions"
    TRAIN_LST:          str = "../data/Bentham/BenthamDatasetR0-GT/Partitions/TrainLines.lst"
    VAL_LST:            str = "../data/Bentham/BenthamDatasetR0-GT/Partitions/ValidationLines.lst"
    TEST_LST:           str = "../data/Bentham/BenthamDatasetR0-GT/Partitions/TestLines.lst"
    MAX_TARGET_LENGTH:  int = 128

@dataclass(frozen=True)
class ModelConfig:
    # trocr-small-handwritten: ~45M params  — faster, lower memory
    # trocr-base-handwritten:  ~334M params — higher accuracy (selected)
    MODEL_NAME: str = "microsoft/trocr-base-handwritten"
    OUTPUT_DIR: str = "../models/bentham_trocr_checkpoint"

## Dataset

The Bentham dataset is already available locally. Directory structure:
```
data/Bentham/BenthamDatasetR0-GT/
├── Images/Lines/          ← cropped line images (.png)
├── Transcriptions/        ← one .txt per image (same stem)
└── Partitions/
    ├── TrainLines.lst
    ├── ValidationLines.lst
    └── TestLines.lst
```

In [ ]:
import tarfile

# Set to True to download automatically when data is missing
DOWNLOAD_IF_MISSING = True

GT_URL     = "http://transcriptorium.eu/~tsdata/BenthamR0/BenthamDatasetR0-GT.tbz"
IMAGES_URL = "http://transcriptorium.eu/~tsdata/BenthamR0/BenthamDatasetR0-Images.tbz"
DATA_BASE  = "../data/Bentham"


def download_and_extract(url: str, dest_dir: str) -> None:
    filename = os.path.join(dest_dir, os.path.basename(url))
    print(f"Downloading {os.path.basename(url)}...", end=" ", flush=True)
    urlretrieve(url, filename)
    print("Done. Extracting...", end=" ", flush=True)
    with tarfile.open(filename, "r:bz2") as tar:
        tar.extractall(dest_dir)
    os.remove(filename)
    print("Done.")


# --- Images ---
if not os.path.isdir(DatasetConfig.IMAGES_DIR):
    if DOWNLOAD_IF_MISSING:
        os.makedirs(DATA_BASE, exist_ok=True)
        download_and_extract(IMAGES_URL, DATA_BASE)
    else:
        raise FileNotFoundError(
            f"Images not found at {DatasetConfig.IMAGES_DIR}\n"
            "Set DOWNLOAD_IF_MISSING = True to download automatically."
        )

# --- Ground-truth transcriptions & partitions ---
if not os.path.isdir(DatasetConfig.TRANSCRIPTIONS_DIR):
    if DOWNLOAD_IF_MISSING:
        os.makedirs(DATA_BASE, exist_ok=True)
        download_and_extract(GT_URL, DATA_BASE)
    else:
        raise FileNotFoundError(
            f"Transcriptions not found at {DatasetConfig.TRANSCRIPTIONS_DIR}\n"
            "Set DOWNLOAD_IF_MISSING = True to download automatically."
        )

# --- Final verification ---
assert os.path.isdir(DatasetConfig.IMAGES_DIR), \
    f"Images dir still missing after download: {DatasetConfig.IMAGES_DIR}"
assert os.path.isdir(DatasetConfig.TRANSCRIPTIONS_DIR), \
    f"Transcriptions dir still missing after download: {DatasetConfig.TRANSCRIPTIONS_DIR}"
assert os.path.isfile(DatasetConfig.TRAIN_LST), \
    f"TrainLines.lst missing: {DatasetConfig.TRAIN_LST}"

print("Dataset ready.")
print(f"  Images:         {DatasetConfig.IMAGES_DIR}")
print(f"  Transcriptions: {DatasetConfig.TRANSCRIPTIONS_DIR}")
print(f"  Partitions:     {os.path.dirname(DatasetConfig.TRAIN_LST)}")

## Parse Transcriptions into DataFrames

Each partition list file contains one line-image stem per line (e.g. `000_000_01`).  
The paired image is `Images/Lines/<stem>.png` and the transcription is `Transcriptions/<stem>.txt`.

In [ ]:
def build_dataframe(partition_lst: str, images_dir: str, transcriptions_dir: str) -> pd.DataFrame:
    """Read a partition list and return a DataFrame with columns [file_name, text]."""
    with open(partition_lst) as f:
        ids = [line.strip() for line in f if line.strip()]

    rows = []
    skipped = 0
    for stem in ids:
        img_path = os.path.join(images_dir, stem + ".png")
        txt_path = os.path.join(transcriptions_dir, stem + ".txt")

        # Skip entries where either file is missing
        if not os.path.exists(img_path) or not os.path.exists(txt_path):
            skipped += 1
            continue

        with open(txt_path, encoding="utf-8") as f:
            text = f.read().strip()

        # Skip empty transcriptions
        if not text:
            skipped += 1
            continue

        rows.append({"file_name": img_path, "text": text})

    df = pd.DataFrame(rows).reset_index(drop=True)
    print(f"  Loaded {len(df)} samples  (skipped {skipped})")
    return df


print("Building splits...")
train_df = build_dataframe(DatasetConfig.TRAIN_LST, DatasetConfig.IMAGES_DIR, DatasetConfig.TRANSCRIPTIONS_DIR)
val_df   = build_dataframe(DatasetConfig.VAL_LST,   DatasetConfig.IMAGES_DIR, DatasetConfig.TRANSCRIPTIONS_DIR)
test_df  = build_dataframe(DatasetConfig.TEST_LST,  DatasetConfig.IMAGES_DIR, DatasetConfig.TRANSCRIPTIONS_DIR)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
train_df.head()

In [ ]:
# Set True to validate the full pipeline quickly on CPU before committing to a long run.
# Set False for real training on the full dataset (or when using a GPU).
QUICK_RUN = False

if QUICK_RUN:
    train_df = train_df.sample(500, random_state=42).reset_index(drop=True)
    val_df   = val_df.sample(100,  random_state=42).reset_index(drop=True)
    print(f"QUICK_RUN active — using {len(train_df)} train / {len(val_df)} val samples")
else:
    print(f"Full dataset — {len(train_df)} train / {len(val_df)} val samples")

## EDA: Explore the Dataset

In [ ]:
# Text length distribution — determines max_target_length
train_df["text_len"] = train_df["text"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train_df["text_len"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Character count distribution (train)")
axes[0].set_xlabel("Characters per line")
axes[0].set_ylabel("Frequency")

axes[1].boxplot(train_df["text_len"], vert=False, patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.6))
axes[1].set_title("Character count box plot (train)")
axes[1].set_xlabel("Characters per line")

plt.tight_layout()
plt.show()

print(train_df["text_len"].describe().round(1))

In [ ]:
# Visualise 8 sample line images with their ground-truth labels
n_samples = 8
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for i in range(n_samples):
    img = Image.open(train_df["file_name"][i]).convert("RGB")
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(train_df["text"][i], fontsize=8, wrap=True)
    axes[i].axis("off")

plt.suptitle("Sample Bentham line images with ground-truth transcriptions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Check image dimensions to understand aspect ratios
widths, heights = [], []
for path in train_df["file_name"][:200]:
    with Image.open(path) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)

print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
print(f"\nNote: TrOCRProcessor resizes images to 384×384 internally.")

## Augmentations

Bentham scans are grayscale with variable ink quality. We apply light augmentations on the training set only to improve robustness without distorting the handwriting.

In [ ]:
train_transforms = transforms.Compose([
    # Bentham images are greyscale scans — keep 3 channels for ViT encoder
    transforms.Grayscale(num_output_channels=3),
    # Small rotations and translations to handle slight skew in scans
    transforms.RandomAffine(degrees=2, translate=(0.02, 0.02)),
    # Brightness/contrast variation to simulate different scan qualities
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
])

# Validation / test: only convert to 3-channel greyscale, no augmentation
eval_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
])

## PyTorch Dataset Class

In [ ]:
class BenthamOCRDataset(Dataset):
    def __init__(self, df: pd.DataFrame, processor: TrOCRProcessor,
                 transforms=None, max_target_length: int = 128):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.transforms = transforms
        self.max_target_length = max_target_length

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        image = Image.open(self.df["file_name"][idx]).convert("RGB")
        if self.transforms:
            image = self.transforms(image)

        pixel_values = self.processor(image, return_tensors="pt").pixel_values

        labels = self.processor.tokenizer(
            self.df["text"][idx],
            padding="max_length",
            max_length=self.max_target_length,
        ).input_ids
        # Replace padding token id with -100 so it is ignored in the loss
        labels = [
            l if l != self.processor.tokenizer.pad_token_id else -100
            for l in labels
        ]

        return {
            "pixel_values": pixel_values.squeeze(),
            "labels": torch.tensor(labels),
        }

## Instantiate Processor and Datasets

In [ ]:
processor = TrOCRProcessor.from_pretrained(ModelConfig.MODEL_NAME)

train_dataset = BenthamOCRDataset(
    df=train_df,
    processor=processor,
    transforms=train_transforms,
    max_target_length=DatasetConfig.MAX_TARGET_LENGTH,
)
val_dataset = BenthamOCRDataset(
    df=val_df,
    processor=processor,
    transforms=eval_transforms,
    max_target_length=DatasetConfig.MAX_TARGET_LENGTH,
)
test_dataset = BenthamOCRDataset(
    df=test_df,
    processor=processor,
    transforms=eval_transforms,
    max_target_length=DatasetConfig.MAX_TARGET_LENGTH,
)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# Sanity check: verify a single sample has the expected shapes
sample = train_dataset[0]
print("pixel_values shape:", sample["pixel_values"].shape)   # expects [3, 384, 384]
print("labels shape:      ", sample["labels"].shape)          # expects [MAX_TARGET_LENGTH]

# Decode the label back to text to confirm the pipeline is wired up correctly
labels = sample["labels"].clone()
labels[labels == -100] = processor.tokenizer.pad_token_id
decoded = processor.tokenizer.decode(labels, skip_special_tokens=True)
print("Decoded label:     ", decoded)

# Show the corresponding image
img = Image.open(train_df["file_name"][0]).convert("RGB")
plt.imshow(img, cmap="gray")
plt.title(f"Label: {train_df['text'][0]}")
plt.axis("off")
plt.show()

## Model Initialization

Load the pre-trained TrOCR encoder-decoder and configure the special token IDs the decoder needs to start and pad generation sequences.

In [ ]:
model = VisionEncoderDecoderModel.from_pretrained(ModelConfig.MODEL_NAME)

# Wire up the decoder's special token IDs required for generation
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

# Gradient checkpointing: trades ~20% speed for ~50% less activation RAM
model.gradient_checkpointing_enable()

model.to(device)
print(f"Model loaded on {device}")

## Metrics: CER & WER

Character Error Rate (CER) and Word Error Rate (WER) are the standard metrics for OCR. `compute_metrics` is passed to the Trainer and called automatically after each evaluation epoch.

In [ ]:
import evaluate

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids   = pred.predictions
    # replace -100 back to pad_token_id so the decoder can process them
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids,   skip_special_tokens=True)
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer, "wer": wer}

## Training

`Seq2SeqTrainer` handles the training loop, gradient accumulation, and checkpointing. `predict_with_generate=True` ensures CER/WER are computed via actual beam-search decoding rather than teacher-forced logits.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=ModelConfig.OUTPUT_DIR,
    num_train_epochs=TrainingConfig.EPOCHS,
    per_device_train_batch_size=TrainingConfig.BATCH_SIZE,
    per_device_eval_batch_size=TrainingConfig.BATCH_SIZE,
    gradient_accumulation_steps=TrainingConfig.GRAD_ACCUM,
    learning_rate=TrainingConfig.LEARNING_RATE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=DatasetConfig.MAX_TARGET_LENGTH,
    generation_num_beams=4,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    logging_steps=100,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## Evaluate on Test Set

In [ ]:
test_results = trainer.evaluate(test_dataset)
print(f"Test CER: {test_results['eval_cer']:.4f}")
print(f"Test WER: {test_results['eval_wer']:.4f}")

## Inference & Error Analysis

Run the trained model on individual test images and compare predictions to ground truth. CER is shown per sample to identify failure cases.

In [ ]:
def predict(image_path: str) -> str:
    image = Image.open(image_path).convert("RGB")
    image = eval_transforms(image)
    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
    generated_ids = model.generate(
        pixel_values,
        num_beams=4,
        max_length=DatasetConfig.MAX_TARGET_LENGTH,
    )
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

# Visualise 8 test samples with ground truth vs prediction
fig, axes = plt.subplots(8, 1, figsize=(14, 24))
for i, ax in enumerate(axes):
    img_path = test_df["file_name"][i]
    gt       = test_df["text"][i]
    pred     = predict(img_path)
    sample_cer = cer_metric.compute(predictions=[pred], references=[gt])
    ax.imshow(Image.open(img_path), cmap="gray")
    ax.set_title(f"GT:   {gt}\nPred: {pred}  (CER={sample_cer:.3f})", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()